# Lab 17 · Build on your gateway with coding agents

Install the AI Gateway plugin in the GitHub Copilot CLI, let the agent discover your gateway's models and MCP tools,
then have it build a ticket summarizer that calls your gateway with its own runtime key. This notebook is the code
track of the [lab guide](README.md): it runs the plugin's slash commands with `copilot -p`, shows every command the
agent ran, then checks the generated code, runs it, and verifies that the agent only read from Azure and that no key
can reach Git. Claude Code, Cursor, and Gemini CLI use the same plugin: follow the guide for them.

**Install the plugin → Discover the gateway → Build an app → Run and verify it**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| Plugin `ai-gateway@azure` | From the [Azure/aigateway](https://github.com/Azure/aigateway) marketplace. It bundles the skill `use-ai-gateway` and the commands `/ai-gateway:discover` and `/ai-gateway:build`. |
| GitHub Copilot CLI | Runs each command with `copilot -p`. The agent loop runs on your machine, with your GitHub Copilot sign-in and models. |
| Azure Resource Manager | Where the plugin reads the gateway, its models, MCP tool servers, and keys, with your `az login` sign-in. It never writes. |
| Model endpoint `<gateway-url>/default/models/openai/v1` | What the generated app calls, with the key `app-coding-agent` in the `api-key` header. |
| `~/ticket-summarizer` | The project folder, a Git repository where the agent writes the app. |
| [../shared/utils.py](../shared/utils.py) | Helpers that every lab notebook shares: settings, your `az login` sign-in, keys, and retries. |

![AI Gateway architecture with coding agents among the clients that call one gateway](../assets/diagrams/architecture.svg)

## Prerequisites

- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`, with read
  access to the gateway and permission to create keys on it.
- [02-first-model-call.ipynb](../02-first-model-call/02-first-model-call.ipynb) has run, so the gateway has the alias
  `gpt-5-mini`.
- Optional: [04-federate-mcp-tools.ipynb](../04-federate-mcp-tools/04-federate-mcp-tools.ipynb) has run, so the
  gateway also has the `learn` MCP server to discover.
- The [GitHub Copilot CLI](https://docs.github.com/en/copilot/how-tos/set-up/install-copilot-cli), tested with
  1.0.106, on the kernel's `PATH` and signed in to GitHub: run `copilot` in a terminal once and use `/login`. The
  notebook's three agent runs use about three premium requests of your Copilot plan.
- Python 3.10 or later with Jupyter.

The notebook installs the plugin in your Copilot CLI and, at clean up, removes it again unless you had it already.
The CLI keeps a session log of each run, as it does for runs you start yourself.

Run the cells in order, or click **Run All**. Each step is idempotent: rerun any cell and it reuses what exists.

In [ ]:
%pip install -q --upgrade azure-identity requests

### 0. Initialize the lab

The defaults match notebooks 01 and 02, so `utils.Lab` finds the gateway without edits. The cell stops here if the
Copilot CLI isn't on the kernel's `PATH`, before the notebook creates anything.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse

import requests
from IPython.display import display

sys.path.insert(0, "../shared")   # helpers and brand styles that every lab notebook shares
import brand
import utils

lab = utils.Lab("17-coding-agent-plugin")
model       = "gpt-5-mini"                         # the alias from lab 02 that the generated app calls
key_name    = "app-coding-agent"                   # the runtime key for this app, so you can revoke it on its own
server_name = "learn"                              # the optional Microsoft Learn MCP server from lab 04
project     = Path.home() / "ticket-summarizer"    # the Git repository where the agent writes the app

copilot = shutil.which("copilot")
assert copilot, "Install the GitHub Copilot CLI and sign in, then restart the kernel so it finds copilot on PATH."
version = subprocess.run([copilot, "--version"], capture_output=True, text=True).stdout.splitlines()[0]
display(brand.note(f"Found {version}", "neutral", copilot))

### 1. Create a key for the app and read the gateway's resource ID

The plugin finds the gateway from its Azure resource ID. The generated app gets its own key, `app-coding-agent`, so
you can revoke it without affecting other apps. The key goes into `AI_GATEWAY_API_KEY` in the environment of the
agent's runs, never into the prompt.

In [ ]:
lab.require(f"{lab.workspace_id}/aliases/{model}", "Run notebook 02 first, or set model to an alias on your gateway.")
has_server = lab.arm("GET", f"{lab.workspace_id}/toolServers/{server_name}") is not None
api_key, created_key = lab.api_key(key_name)
display(brand.key(key_name, api_key, created_key))

agent_env = {
    **os.environ,
    "PATH":               f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}",   # python is the kernel's
    "AI_GATEWAY_URL":     lab.gateway_url,
    "AI_GATEWAY_MODEL":   model,
    "AI_GATEWAY_API_KEY": api_key,
}
display(brand.info("What you give the coding agent", [
    ("Gateway resource ID", brand.mono(lab.gateway_id)),
    ("AI_GATEWAY_URL",      brand.mono(lab.gateway_url)),
    ("AI_GATEWAY_MODEL",    brand.mono(model)),
    ("AI_GATEWAY_API_KEY",  brand.masked(api_key)),
    ("MCP server learn",    brand.chip("Published", "ok") if has_server else brand.chip("Not published", "neutral")),
]))

### 2. Create the project folder

The agent works in `~/ticket-summarizer`, a Git repository, so step 7 can check that no secret can be committed.
The cell adds `ticket.txt`, a sample support ticket for the app to summarize.

In [ ]:
created_project = not project.exists()
if created_project:
    project.mkdir(parents=True)
    subprocess.run(["git", "init", "-q"], cwd=project, check=True)
    (project / "ticket.txt").write_text(
        "Subject: Checkout fails with error 502 since this morning\n\n"
        "Hi team, since about 8:00 UTC our customers can't complete checkout on the EU store. The payment page spins "
        "for 30 seconds and then shows \"502 Bad Gateway\". The US store works. We didn't deploy anything today. "
        "About 40% of EU orders are failing, and our support queue has 120 new tickets. Please treat this as urgent.\n\n"
        "Order example: EU-77812. Contact: Dana Ruiz, on-call lead.\n")
display(brand.note(f"{'Created' if created_project else 'Reusing'} the project folder", "ok", project))

### 3. Install the plugin

The plugin comes from the marketplace `azure`, the [Azure/aigateway](https://github.com/Azure/aigateway)
repository. The cell runs the CLI's `plugin` subcommands, the shell form of `/plugin marketplace add` and
`/plugin install`, and records what was already there, so **Clean up** removes only what this notebook added.

In [ ]:
def cli(*args):
    """Run a copilot subcommand and return its output. Stop if it fails."""
    done = subprocess.run([copilot, *args], capture_output=True, text=True, timeout=300)
    assert done.returncode == 0, (done.stderr or done.stdout)[-2000:]
    return done.stdout


added_marketplace = not re.search(r"^\s*\W\s*azure \(", cli("plugin", "marketplace", "list"), re.M)
if added_marketplace:
    cli("plugin", "marketplace", "add", "Azure/aigateway")
installed_plugin = "ai-gateway@azure" not in cli("plugin", "list")
if installed_plugin:
    cli("plugin", "install", "ai-gateway@azure")

installed = re.search(r"ai-gateway@azure \((v[^)]+)\)", cli("plugin", "list"))
assert installed, "The plugin isn't installed. Rerun the cell."
display(brand.table("The AI Gateway plugin in your Copilot CLI", ["Item", "Name", "Status"], [
    ("Marketplace", brand.mono("azure"), brand.chip("Added" if added_marketplace else "Already registered", "ok")),
    ("Plugin", brand.mono(f"ai-gateway@azure {installed[1]}"),
     brand.chip("Installed" if installed_plugin else "Already installed", "ok")),
    ("Skill", brand.mono("use-ai-gateway"), brand.chip("Loaded on demand", "neutral")),
    ("Commands", brand.mono("/ai-gateway:discover · /ai-gateway:build"), brand.chip("Available", "neutral")),
]))

### 4. Check that the commands are available

`run_copilot` runs one prompt with `copilot -p` in the project folder, and reads what happened from the CLI's JSON
events: the answer, the commands the agent ran, and the premium requests. `copilot -p` can't ask for approval, so
each run gets its permissions up front, and the CLI denies anything else.

Run `/ai-gateway:discover` without a resource ID, and without permissions. The plugin answers that it needs the
gateway's resource ID. That's how you know it's loaded.

In [ ]:
host = urlparse(lab.gateway_url).hostname
read_only = [   # what discovery needs: read files in the project, run az, curl, jq, Python, and printing commands,
                # and reach ARM and the gateway
    f"--allow-tool=read({project}/**)", "--allow-tool=shell(az:*)", "--allow-tool=shell(curl:*)",
    "--allow-tool=shell(jq:*)", "--allow-tool=shell(command:*)", "--allow-tool=shell(python:*)",
    "--allow-tool=shell(python3:*)", "--allow-tool=shell(printf:*)", "--allow-tool=shell(echo:*)",
    "--allow-url=management.azure.com", f"--allow-url={host}",
]
WRITES = re.compile(r"(-X|--request|--method|-m)\s*['\"]?(PUT|PATCH|DELETE)\b|method\s*=\s*['\"](PUT|PATCH|DELETE)"
                    r"|requests\.(put|patch|delete)\(|\baz\s+(?!rest\b)[\w -]*?\b(create|update|delete|regenerate)\b",
                    re.I)


def mask(text):
    """Hide the key and any bearer token in a command before it's shown."""
    text = text.replace(api_key, "<AI_GATEWAY_API_KEY>")
    return re.sub(r"eyJ[\w-]+\.[\w-]+\.[\w-]+", "<token>", text)


def touches(command):
    """Say where a shell command goes: Azure (reads or writes), the gateway, or only your machine."""
    if "management.azure.com" in command or re.search(r"(^|[\s;&|(])az\s+[a-z]", command):
        return "Azure, writes" if WRITES.search(command) else "Azure, reads"
    return "Gateway" if host in command else "Your machine"


def run_copilot(prompt, *flags):
    """Run copilot -p in the project folder, and return what happened, read from its JSON events."""
    status = brand.live(brand.note(f"Copilot CLI: {mask(prompt)[:110]}", "wait"))
    start = time.time()
    done = subprocess.run([copilot, "-p", prompt, *flags, "--no-ask-user", "--output-format", "json"],
                          cwd=project, env=agent_env, capture_output=True, text=True, timeout=900)
    events = [json.loads(line) for line in done.stdout.splitlines() if line.startswith("{")]

    def data(kind):
        return [event.get("data", event) for event in events if event["type"] == kind]

    answers = [m for m in data("assistant.message") if m.get("content")]
    text = [line for line in (done.stderr + done.stdout).splitlines() if line.strip() and not line.startswith("{")]
    run = {
        "exit_code": done.returncode,
        "answer":    answers[-1]["content"] if answers else "",
        "model":     answers[-1].get("model") if answers else None,
        "commands":  [(tool["arguments"].get("description", ""), tool["arguments"]["command"])
                      for tool in data("tool.execution_start") if tool.get("toolName") == "bash"],
        "tools":     [tool["toolName"] for tool in data("tool.execution_start")],
        "denied":    sum("Permission denied" in str(tool.get("error")) for tool in data("tool.execution_complete")),
        "error":     next((error["message"] for error in data("session.error")), " ".join(text)[:300]),
        "premium":   next((result["usage"].get("premiumRequests") for result in data("result") if "usage" in result), None),
        "seconds":   round(time.time() - start),
    }
    status.update(brand.note(f"Copilot CLI finished in {run['seconds']} s with exit code {run['exit_code']}",
                             "neutral" if run["exit_code"] == 0 else "warn"))
    assert run["exit_code"] == 0, run["error"]
    return run


def show_run(run, title, badge=None):
    """Show the agent's answer, then each shell command it ran and where the command went."""
    premium = brand.chip(f"{run['premium']} premium requests", "neutral") if run["premium"] is not None else None
    display(brand.reply(run["answer"], title, model=run["model"], badge=brand.chips(*[b for b in (badge, premium) if b]),
                        markdown=True,
                        sub=f"{len(run['tools'])} tool calls in {run['seconds']} s"
                            + (f", {run['denied']} denied" if run["denied"] else "")))
    tones = {"Azure, reads": "ok", "Azure, writes": "bad", "Gateway": "brand", "Your machine": "neutral"}
    if run["commands"]:
        display(brand.table("Commands the agent ran", ["What for", "Command", "Goes to"], [
            (description, brand.mono(mask(command.strip().splitlines()[0])[:90]), brand.chip(touches(command), tones[touches(command)]))
            for description, command in run["commands"]]))


check = run_copilot("/ai-gateway:discover")
asks = re.search(r"resource id|/subscriptions/", check["answer"], re.I)
show_run(check, "/ai-gateway:discover · no resource ID",
         badge=brand.chip("Asks for the resource ID", "ok") if asks else brand.chip("No request for an ID", "warn"))

### 5. Discover what the gateway offers

Now pass the resource ID, with read-only permissions: `az`, `curl`, `jq`, Python, `printf`, and `echo`, and
network access to Azure Resource Manager and your gateway only. The plugin reads the runtime URL, lists the models
with the name each one accepts, and checks each MCP tool server with a full MCP handshake. The prompt tells the
plugin to use the key in `AI_GATEWAY_API_KEY` for the handshake, so it doesn't read any key's secret. The table
shows that every Azure command only reads. Discovery takes about a minute.

In [ ]:
discovery = run_copilot(f"/ai-gateway:discover {lab.gateway_id} For the MCP handshake, use the key in the "
                        "AI_GATEWAY_API_KEY environment variable.", *read_only)
found = [name for name in [model, *([server_name] if has_server else [])] if name in discovery["answer"]]
show_run(discovery, "/ai-gateway:discover · your gateway",
         badge=brand.chip(f"Finds {', '.join(found)}", "ok") if found else brand.chip(f"No {model}", "warn"))
assert model in discovery["answer"], f"The summary doesn't list {model}. Rerun the cell."

### 6. Build an app with the gateway

`build` discovers again, integrates the model into the project, and runs the code once. The description names the
model and the key's environment variable, so the plugin doesn't list keys or read their secrets. This run also
needs to write files, but only in the project folder. It takes one to two minutes.

In [ ]:
task = (f"/ai-gateway:build {lab.gateway_id} a Python command-line script named summarize_ticket.py that summarizes "
        f"the support ticket in the text file passed as its first argument, such as ticket.txt. Use the {model} model "
        "and read the key from the AI_GATEWAY_API_KEY environment variable. This run is non-interactive, so don't ask "
        "questions: decide, build, and run it once.")
build = run_copilot(task, *read_only, f"--allow-tool=write({project}/**)")
show_run(build, "/ai-gateway:build · ticket summarizer")

status = subprocess.run(["git", "status", "--porcelain", "--untracked-files=all"], cwd=project,
                        capture_output=True, text=True, check=True).stdout.splitlines()
files = sorted(line[3:] for line in status if line[3:] != "ticket.txt")
rows = [(brand.mono(name), f"{(project / name).stat().st_size:,} bytes", brand.chip("Untracked", "brand"))
        for name in files]
if (project / ".env").exists() and ".env" not in files:
    rows.append((brand.mono(".env"), f"{(project / '.env').stat().st_size:,} bytes", brand.chip("Ignored", "neutral")))
display(brand.table("Files the agent added", ["File", "Size", "Git"], rows, sub=str(project)))
assert (project / "summarize_ticket.py").exists(), "The agent didn't write summarize_ticket.py. Rerun the cell."

### 7. Review the generated code

Read the code before you run it again or commit it. The cell shows it, then checks what the
[lab guide](README.md#review-the-generated-code) asks you to check: the endpoint, the model name, where the key
comes from, and that the app calls only the gateway.

In [ ]:
source = (project / "summarize_ticket.py").read_text()
review = [
    ("Calls the model endpoint under /default/", "/default/models/openai/v1" in source, ""),
    (f"Uses the {model} model", model in source, ""),
    ("Reads the key from AI_GATEWAY_API_KEY", "AI_GATEWAY_API_KEY" in source, ""),
    ("Holds no key value", api_key not in source, ""),
    ("Calls only the gateway, never Azure Resource Manager", "management.azure.com" not in source, ""),
]
display(brand.card("summarize_ticket.py", brand.markdown(f"```python\n{source.strip()}\n```"),
                   sub=f"{len(source.splitlines())} lines, written by the coding agent"))
display(brand.table("Review the generated code", ["Check", "Result"], [
    (label, brand.chip("Pass", "ok") if ok else brand.chip("Fail", "bad")) for label, ok, _ in review]))
assert all(ok for _, ok, _ in review), "Read the code and the checks above."

### 8. Run the app

Run the app yourself with the kernel's Python and the key in `AI_GATEWAY_API_KEY`, as you would from your shell. If
the agent added a `requirements.txt`, the cell installs it first. A new key can take a minute to work, so the cell
retries on `401`.

In [ ]:
if (project / "requirements.txt").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=project, check=True)


def run_app():
    """Run the generated app on ticket.txt, and return its exit code and output."""
    done = subprocess.run([sys.executable, "summarize_ticket.py", "ticket.txt"], cwd=project, env=agent_env,
                          capture_output=True, text=True, timeout=300)
    return done.returncode, done.stdout.strip() or done.stderr.strip()


for attempt in range(5):
    exit_code, output = run_app()
    if exit_code == 0 or "401" not in output:
        break
    time.sleep(30)
display(brand.reply(output, "python summarize_ticket.py ticket.txt", model=model, markdown=True,
                    sub="The generated app, through your gateway",
                    badge=brand.chip("Exit code 0", "ok") if exit_code == 0 else brand.chip(f"Exit code {exit_code}", "bad")))
assert exit_code == 0, output

### 9. Verify

The cell runs the guide's checks, and two that the guide leaves to you when you approve commands:

- A direct call with the key and the alias returns `200`, so the gateway setup works outside the generated code.
- `.env`, if the agent wrote one, is ignored by Git, and your key is in no other project file.
- Every command that the agent ran against Azure only read: no `PUT`, `PATCH`, or `DELETE` request, and no `az`
  command that creates, updates, deletes, or regenerates.
- No run read a key's secret, because the prompts named the key in `AI_GATEWAY_API_KEY`.

In [ ]:
direct = requests.post(f"{lab.gateway_url}/default/models/openai/v1/chat/completions", timeout=120,
                       headers={"api-key": api_key},
                       json={"model": model, "messages": [{"role": "user", "content": "ping"}]}).status_code

skip = {".git", ".venv", "node_modules"}
holders = [str(path.relative_to(project)) for path in project.rglob("*")
           if path.is_file() and path.name != ".env" and not skip & set(path.relative_to(project).parts)
           and api_key in path.read_text(errors="ignore")]
env_file = project / ".env"
ignored = subprocess.run(["git", "check-ignore", "-q", ".env"], cwd=project).returncode == 0
azure = Counter(touches(command) for run in (check, discovery, build) for _, command in run["commands"])
secrets = sum(bool(re.search(r"listSecrets|list-secrets", command, re.I))
              for run in (check, discovery, build) for _, command in run["commands"])

verify = [
    ("Direct model call returns HTTP 200", direct == 200, f"HTTP {direct}"),
    ("No .env file, or Git ignores it", not env_file.exists() or ignored,
     "No .env file" if not env_file.exists() else ".env is ignored" if ignored else ".env is not ignored"),
    ("No key in other project files", not holders, ", ".join(holders) or "None"),
    ("The agent only read from Azure", not azure["Azure, writes"],
     f"{azure['Azure, reads']} reads, {azure['Azure, writes']} writes"),
    ("The agent read no key's secret", not secrets, f"{secrets} secret reads"),
]
display(brand.table("Verify", ["Check", "Result", "Detail"], [
    (label, brand.chip("Pass", "ok") if ok else brand.chip("Fail", "bad"), brand.mono(detail))
    for label, ok, detail in verify], sub=str(project),
    badge=brand.chip(f"{sum(ok for _, ok, _ in verify)} of {len(verify)} passed", "ok")))
assert all(ok for _, ok, _ in verify)

### 10. Clean up resources

The cell deletes what this notebook created: the `app-coding-agent` key, which stops working everywhere, and the
project folder, because a `.env` file in it can hold a live key. It also removes the plugin and the marketplace from
your Copilot CLI if the notebook added them. It keeps the gateway and the alias, because other labs reuse them.

In [ ]:
if created_project:
    shutil.rmtree(project)
    display(brand.note(f"Deleted {project}", "ok"))
if created_key:
    lab.delete(f"{lab.gateway_id}/apiKeys/{key_name}")
if installed_plugin:
    cli("plugin", "uninstall", "ai-gateway")
    display(brand.note("Removed the plugin ai-gateway from your Copilot CLI", "ok"))
if added_marketplace:
    cli("plugin", "marketplace", "remove", "azure")
    display(brand.note("Removed the marketplace azure from your Copilot CLI", "ok"))

kept = [name for name, created in [(key_name, created_key), (str(project), created_project),
                                   ("ai-gateway@azure", installed_plugin), ("marketplace azure", added_marketplace)]
        if not created]
if kept:
    display(brand.note(f"Kept what existed before this notebook ran: {', '.join(kept)}", "neutral"))

## Troubleshooting

- **Step 0 stops with `Install the GitHub Copilot CLI`**: `shutil.which("copilot")` didn't find the CLI. Install
  it, check that `copilot --version` works in a terminal, then restart the editor so the kernel gets the new `PATH`.
- **`Error: No authentication information found.`**: the CLI isn't signed in. Run `copilot` in a terminal and use
  `/login`, or run `gh auth login`, then rerun the cell.
- **`n denied` under the agent's answer**: the agent tried a tool that the run's permissions don't allow, for example
  `rm`, and the CLI refused it. The agent usually finds another way. If the answer says it couldn't finish, rerun the
  cell, or run the same command in an interactive `copilot` session where you approve each tool.
- **Step 5 doesn't find the model**: check the alias in the portal, or set `model` in step 0 to an alias that your
  gateway publishes. The answer differs between runs, so rerun the cell once.
- **The app returns `HTTP 401`**: a new key can take a minute to work. Rerun step 8.
- **The app returns `HTTP 404`**: the generated URL misses `/default/`. Ask the agent to fix it, or rerun step 6.

For the other errors in this lab, see [Troubleshooting](../TROUBLESHOOTING.md#lab-17-coding-agent-plugin).

## Next steps

- Use the plugin in Claude Code, Cursor, or Gemini CLI, and give VS Code agent mode the gateway's tools, with the
  [lab guide](README.md).
- Run GitHub Copilot itself on your gateway's models and tools in [lab 24](../24-github-copilot-byok/README.md) and
  its notebook [24-github-copilot-byok.ipynb](../24-github-copilot-byok/24-github-copilot-byok.ipynb).
- Connect agent frameworks to the same models and tools in [lab 15](../15-agent-frameworks/README.md) and its
  notebook [15-agent-frameworks.ipynb](../15-agent-frameworks/15-agent-frameworks.ipynb).
- Docs: [Use the AI Gateway plugin](https://ai.gateway.azure.com/docs/quickstarts/skills) and
  [Finding and installing plugins for GitHub Copilot CLI](https://docs.github.com/en/copilot/how-tos/copilot-cli/customize-copilot/plugins-finding-installing).
- Read this lab on the labs site: [https://azure.github.io/aigateway/labs/17-coding-agent-plugin/](https://azure.github.io/aigateway/labs/17-coding-agent-plugin/).